# Nemotron LoRA Training — Fully Offline, RTX Pro 6000

**Required Kaggle inputs:**
- `nemotron-offline-deps` — wheelhouse + NuminaMath parquet
- `huikang/nemotron-adapter` — competition LoRA adapter
- `nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16` — base model

**Internet must be OFF.**

**Key fixes vs previous version:**
- Cell 6: `load_best_model_at_end` removed (was triggering OOM by reloading full model into VRAM at end of training)
- Cell 6: auto-resumes from latest checkpoint if session was previously interrupted
- Cell 6: `warmup_ratio` replaced with `warmup_steps` (deprecation warning)
- Cell 8: replaced `model.save_pretrained()` with direct file copy from checkpoint — eliminates OOM
- Cell 9: uses `zipfile` directly to guarantee adapter files sit at zip root

## Cell 1 — Install packages from local wheels (offline)

In [ ]:
import os, glob, json, subprocess, sys, re, shutil, site
import importlib.metadata as imd

def find_wheelhouse():
    for dirpath, _, files in os.walk('/kaggle/input'):
        if any(f.endswith('.whl') for f in files):
            return dirpath
    return None

WHEEL_DIR = find_wheelhouse()
assert WHEEL_DIR, 'Wheelhouse not found'
print(f'Wheelhouse: {WHEEL_DIR}')

with open(os.path.join(WHEEL_DIR, 'manifest.json')) as f:
    manifest = json.load(f)
print('manifest:', json.dumps(manifest, indent=2))

all_wheels = sorted(glob.glob(WHEEL_DIR + '/*.whl'))
print(f'Found {len(all_wheels)} wheels.')

import numpy as _np
MEM_NUMPY = _np.__version__
print(f'numpy in memory: {MEM_NUMPY}')


def install(spec, no_deps=False, force=False):
    cmd = [sys.executable, '-m', 'pip', 'install',
           '--no-index', f'--find-links={WHEEL_DIR}', spec, '-q']
    if no_deps: cmd.append('--no-deps')
    if force:   cmd.append('--force-reinstall')
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        stderr = r.stderr.strip()
        if 'No matching distribution' in stderr and '==' in spec:
            pkg_name = spec.split('==')[0]
            print(f'  WARN: {spec} not in wheelhouse — retrying {pkg_name} (latest)')
            cmd2 = [sys.executable, '-m', 'pip', 'install',
                    '--no-index', f'--find-links={WHEEL_DIR}', pkg_name, '-q']
            if no_deps: cmd2.append('--no-deps')
            if force:   cmd2.append('--force-reinstall')
            r2 = subprocess.run(cmd2, capture_output=True, text=True)
            if r2.returncode == 0:
                try:    ver = imd.version(pkg_name)
                except: ver = '?'
                print(f'  OK: {pkg_name} (installed: {ver})')
                return True
            print(f'  ERROR {pkg_name}: {r2.stderr.strip()[:300]}')
            return False
        print(f'  ERROR {spec}: {stderr[:300]}')
        return False
    try:    ver = imd.version(spec.split('==')[0])
    except: ver = '?'
    print(f'  OK: {spec} (installed: {ver})')
    return True


for pkg in ['scikit-learn', 'scipy']:
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'uninstall', pkg, '-y', '-q'],
        capture_output=True, text=True)
    print(f'{pkg}: {"removed" if r.returncode == 0 else "already absent"}')

print('\nInstalling packages...')

unsloth_zoo_ver = manifest.get('unsloth_zoo', '')
install(f'unsloth_zoo=={unsloth_zoo_ver}' if unsloth_zoo_ver else 'unsloth_zoo')

unsloth_ver = manifest.get('unsloth', '')
install(f'unsloth=={unsloth_ver}' if unsloth_ver else 'unsloth', no_deps=True)

for pkg in ['huggingface_hub', 'bitsandbytes', 'accelerate', 'datasets', 'peft']:
    ver = manifest.get(pkg.lower(), '')
    install(f'{pkg}=={ver}' if ver else pkg)

install('trl', no_deps=True)

tf_ver = manifest.get('transformers', '5.5.0')
install(f'transformers=={tf_ver}', force=True)

for pattern, pip_name in [('causal_conv1d*.whl', 'causal-conv1d'),
                           ('mamba_ssm*.whl',     'mamba-ssm')]:
    matches = glob.glob(os.path.join(WHEEL_DIR, pattern))
    if matches:
        print(f'  Installing {pip_name}: {os.path.basename(matches[0])}')
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'install',
             matches[0], '--no-deps', '--no-index', '--force-reinstall', '-q'],
            capture_output=True, text=True)
        print(f'  {"OK" if r.returncode == 0 else "ERROR: " + r.stderr[:300]}')
    else:
        print(f'  WARNING: no wheel found for {pip_name}')

import importlib
importlib.invalidate_caches()

search_dirs = list(site.getsitepackages())
try:
    search_dirs.append(site.getusersitepackages())
except Exception:
    pass

numpy_distinfos = []
for d in search_dirs:
    numpy_distinfos.extend(glob.glob(os.path.join(d, 'numpy-*.dist-info')))
numpy_distinfos = sorted(set(numpy_distinfos))
print(f'\nnumpy dist-info dirs found: {[os.path.basename(d) for d in numpy_distinfos]}')

patched = False
for dist_dir in numpy_distinfos:
    disk_ver = os.path.basename(dist_dir).replace('numpy-', '').replace('.dist-info', '')
    if disk_ver == MEM_NUMPY:
        print(f'dist-info already matches in-memory version ({MEM_NUMPY}) — no patch needed.')
        patched = True
        break
    metadata_path = os.path.join(dist_dir, 'METADATA')
    if not os.path.exists(metadata_path):
        continue
    meta = open(metadata_path).read()
    meta_new = re.sub(r'^Version:.*$', f'Version: {MEM_NUMPY}', meta, flags=re.MULTILINE)
    open(metadata_path, 'w').write(meta_new)
    new_dir = os.path.join(os.path.dirname(dist_dir), f'numpy-{MEM_NUMPY}.dist-info')
    if os.path.exists(new_dir):
        shutil.rmtree(new_dir)
    os.rename(dist_dir, new_dir)
    print(f'Patched: {os.path.basename(dist_dir)} -> numpy-{MEM_NUMPY}.dist-info')
    patched = True
    break

if not patched:
    print(f'No numpy dist-info found — creating synthetic one for {MEM_NUMPY}')
    target_dir = os.path.join(site.getsitepackages()[0], f'numpy-{MEM_NUMPY}.dist-info')
    os.makedirs(target_dir, exist_ok=True)
    with open(os.path.join(target_dir, 'METADATA'), 'w') as fh:
        fh.write(f'Metadata-Version: 2.1\nName: numpy\nVersion: {MEM_NUMPY}\n')
    with open(os.path.join(target_dir, 'INSTALLER'), 'w') as fh:
        fh.write('pip\n')
    print(f'Created: {target_dir}')
    patched = True

importlib.invalidate_caches()
import importlib.metadata as imd2

disk_numpy_now = imd2.version('numpy')
print(f'numpy in memory  : {MEM_NUMPY}')
print(f'numpy on disk now: {disk_numpy_now}')
assert disk_numpy_now == MEM_NUMPY, f'Patch failed: disk shows {disk_numpy_now}'
print('numpy version match confirmed.')

print('\nFinal versions:')
for pkg in ['numpy', 'unsloth', 'unsloth_zoo', 'trl', 'peft',
            'transformers', 'accelerate', 'bitsandbytes',
            'mamba_ssm', 'causal_conv1d']:
    try:
        print(f'  {pkg}: {imd2.version(pkg)}')
    except Exception:
        print(f'  {pkg}: NOT FOUND')

import unsloth
import mamba_ssm, causal_conv1d
print(f'unsloth       : {unsloth.__version__}')
print(f'mamba_ssm     : {mamba_ssm.__version__}')
print(f'causal_conv1d : {causal_conv1d.__version__}')
print('\nCell 1 complete.')


## Cell 2 — Detect model type & locate base model

In [ ]:
import os, json, shutil

COMPETITION_PATH = '/kaggle/input/models/huikang/nemotron-adapter/transformers/default/20'

files_here = os.listdir(COMPETITION_PATH) if os.path.isdir(COMPETITION_PATH) else []
print('Files at competition path:')
for f in sorted(files_here):
    print(f'  {f}')

IS_ADAPTER = 'adapter_config.json' in files_here
IS_BASE    = 'config.json' in files_here and not IS_ADAPTER
print(f'\nPath type: {"LoRA adapter" if IS_ADAPTER else "base model" if IS_BASE else "UNKNOWN"}')

BASE_MODEL_PATH = None
ADAPTER_PATH    = None

if IS_BASE:
    BASE_MODEL_PATH = COMPETITION_PATH
    print('Competition path is a base model — using directly.')

elif IS_ADAPTER:
    with open(os.path.join(COMPETITION_PATH, 'adapter_config.json')) as f:
        adapter_cfg = json.load(f)
    print('\nadapter_config.json:')
    print(json.dumps(adapter_cfg, indent=2))

    print('\nSearching for base model in /kaggle/input...')
    candidates = []
    for dirpath, _, dirfiles in os.walk('/kaggle/input'):
        if COMPETITION_PATH in dirpath:
            continue
        if 'config.json' in dirfiles and 'adapter_config.json' not in dirfiles:
            has_weights = any(
                f.endswith('.safetensors') or f.endswith('.bin') for f in dirfiles
            )
            if has_weights:
                try:
                    with open(os.path.join(dirpath, 'config.json')) as f:
                        cfg = json.load(f)
                    candidates.append((dirpath, cfg.get('model_type', '?')))
                except Exception:
                    pass

    print(f'Candidates found: {len(candidates)}')
    for p, mt in candidates:
        print(f'  [{mt}] {p}')

    if not candidates:
        raise RuntimeError(
            'No base model found in /kaggle/input!\n'
            'Add nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16 as a Kaggle model input.'
        )

    BASE_MODEL_PATH = candidates[0][0]
    print(f'\nSelected base model: {BASE_MODEL_PATH}')

    current_ref = adapter_cfg.get('base_model_name_or_path')
    if not current_ref or str(current_ref).lower() in ('none', 'null', ''):
        print(f'Patching adapter_config: base_model_name_or_path was {current_ref!r}')
        ADAPTER_COPY = '/kaggle/working/adapter_patched'
        if os.path.exists(ADAPTER_COPY):
            shutil.rmtree(ADAPTER_COPY)
        shutil.copytree(COMPETITION_PATH, ADAPTER_COPY)
        patched_cfg = dict(adapter_cfg)
        patched_cfg['base_model_name_or_path'] = BASE_MODEL_PATH
        with open(os.path.join(ADAPTER_COPY, 'adapter_config.json'), 'w') as f:
            json.dump(patched_cfg, f, indent=2)
        ADAPTER_PATH = ADAPTER_COPY
        print(f'Patched adapter written to: {ADAPTER_PATH}')
    else:
        print(f'base_model_name_or_path already set: {current_ref}')
        ADAPTER_PATH = COMPETITION_PATH

else:
    raise RuntimeError(
        f'Competition path is neither a base model nor a LoRA adapter.\n'
        f'Path: {COMPETITION_PATH}\n'
        f'Files found: {files_here}'
    )

print(f'\nBASE_MODEL_PATH = {BASE_MODEL_PATH}')
print(f'ADAPTER_PATH    = {ADAPTER_PATH}')


## Cell 3 — Load base model + competition adapter (4-bit QLoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048
LORA_RANK   = 32

print(f'Loading base model: {BASE_MODEL_PATH}')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_PATH,
    max_seq_length=MAX_SEQ_LEN,
    dtype=torch.bfloat16,
    load_in_4bit=True,
    trust_remote_code=True,
)
print(f'Base loaded. VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

# MoE expert dtype-safety hooks: survive all for_training() re-entries
def _cast_to_bf16_hook(module, args, output):
    if isinstance(output, torch.Tensor) and output.dtype != torch.bfloat16:
        return output.to(torch.bfloat16)
    return output

def register_moe_hooks(mdl):
    hooks = []
    for name, module in mdl.named_modules():
        parent = name.rsplit('.', 1)[0] if '.' in name else ''
        if 'experts' in parent and not list(module.children()):
            hooks.append(module.register_forward_hook(_cast_to_bf16_hook))
    return hooks

if ADAPTER_PATH:
    print(f'\nLoading competition adapter: {ADAPTER_PATH}')
    model.load_adapter(ADAPTER_PATH, adapter_name='default', is_trainable=True)

    cast_count = 0
    for name, param in model.named_parameters():
        if param.requires_grad and param.dtype != torch.bfloat16:
            param.data = param.data.to(torch.bfloat16)
            cast_count += 1
    print(f'Cast {cast_count} trainable LoRA parameter tensors -> bfloat16.')

    model = FastLanguageModel.for_training(model)
    print('Competition adapter loaded.')

else:
    print('No adapter path — applying fresh LoRA.')
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        lora_alpha=64,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                        'gate_proj', 'up_proj', 'down_proj'],
        lora_dropout=0.0,
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=42,
    )
    for name, param in model.named_parameters():
        if param.requires_grad and param.dtype != torch.bfloat16:
            param.data = param.data.to(torch.bfloat16)
    print('Fresh LoRA applied.')

_moe_hooks = register_moe_hooks(model)
print(f'Registered {len(_moe_hooks)} MoE expert output hooks (bfloat16 guard).')

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'\nTrainable : {trainable / 1e6:.1f} M  ({100 * trainable / total:.2f}%)')
print(f'VRAM      : {torch.cuda.memory_allocated() / 1e9:.1f} GB')


## Cell 4 — Load NuminaMath dataset from local parquet (offline)

In [ ]:
from datasets import load_dataset
import os

def find_parquet(hint='numinamath'):
    for dirpath, _, files in os.walk('/kaggle/input'):
        if hint.lower() in dirpath.lower() or any(hint in f for f in files):
            for f in files:
                if f.endswith('.parquet'):
                    return os.path.join(dirpath, f)
    for dirpath, _, files in os.walk('/kaggle/input'):
        for f in files:
            if f.endswith('.parquet'):
                return os.path.join(dirpath, f)
    return None

PARQUET_PATH = find_parquet()
assert PARQUET_PATH, 'NuminaMath parquet not found — check nemotron-offline-deps dataset is attached.'
print(f'Loading: {PARQUET_PATH}')

ds = load_dataset('parquet', data_files={'train': PARQUET_PATH}, split='train')
print(f'Loaded  : {len(ds):,} examples')
print(f'Columns : {ds.column_names}')
print(f'\nSample problem:\n{ds[0]["problem"][:300]}')
print(f'\nSample solution (first 200 chars):\n{ds[0]["solution"][:200]}')


## Cell 5 — Format dataset into chat template

In [ ]:
SYSTEM_PROMPT = (
    'You are a mathematical reasoning expert. '
    'Solve problems step by step, showing all working. '
    'Always place your final answer inside \\boxed{} at the end.'
)

def format_for_sft(example):
    messages = [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': example['problem']},
        {'role': 'assistant', 'content': example['solution']},
    ]
    return {
        'text': tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    }

N_SFT = min(50_000, len(ds))
sft_raw  = ds.select(range(N_SFT))
grpo_raw = ds.select(range(N_SFT, len(ds)))

print(f'SFT pool  : {len(sft_raw):,} examples')
print(f'GRPO pool : {len(grpo_raw):,} examples')

ds_fmt = sft_raw.map(
    format_for_sft,
    num_proc=2,
    remove_columns=sft_raw.column_names,
    desc='Formatting',
)
print(f'\nFormatted : {len(ds_fmt):,} examples')
print(f'\nSample (first 500 chars):')
print(ds_fmt[0]['text'][:500])


## Cell 6 — SFT Training

Auto-resumes from the latest checkpoint if a previous session was interrupted.
`load_best_model_at_end` is intentionally omitted — reloading the best model at the end of training caused the OOM crash by allocating a second copy of the model in VRAM. Cell 8 finds the best checkpoint independently via `trainer_state.json`.

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
import torch, os, glob, json

# BFloat16RecastCallback: fires after unsloth's internal for_training(), re-casts
# any LoRA params that got reset to Float32 back to bfloat16.
class BFloat16RecastCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, model=None, **kwargs):
        if model is None:
            return
        cast_count = 0
        for name, param in model.named_parameters():
            if param.requires_grad and param.dtype != torch.bfloat16:
                param.data = param.data.to(torch.bfloat16)
                cast_count += 1
        if cast_count:
            print(f'[BFloat16RecastCallback] Re-cast {cast_count} LoRA params -> bfloat16.')

# Auto-resume from latest checkpoint if this session was previously interrupted
CKPT_DIR = '/kaggle/working/checkpoints'
resume_from = None
if os.path.isdir(CKPT_DIR):
    existing = sorted(glob.glob(f'{CKPT_DIR}/checkpoint-*'))
    if existing:
        resume_from = existing[-1]
        print(f'Found existing checkpoint — will resume from: {resume_from}')
    else:
        print('Checkpoint dir exists but empty — starting fresh.')
else:
    print('No checkpoint dir — starting fresh.')

split    = ds_fmt.train_test_split(test_size=0.01, seed=42)
train_ds = split['train']
eval_ds  = split['test']
print(f'Train : {len(train_ds):,}  |  Eval : {len(eval_ds):,}')

sft_args = SFTConfig(
    output_dir=CKPT_DIR,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=2,

    num_train_epochs=2,
    warmup_steps=100,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',

    optim='adamw_8bit',
    weight_decay=0.01,
    max_grad_norm=1.0,

    eval_strategy='steps',
    eval_steps=200,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=3,
    # load_best_model_at_end=False (default) — avoids a full model reload into
    # VRAM at end of training which caused the OOM crash.

    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field='text',
    packing=True,

    bf16=True,
    tf32=True,

    logging_steps=25,
    report_to='none',
    seed=42,
    dataloader_num_workers=2,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=sft_args,
    callbacks=[BFloat16RecastCallback()],
)

# Patch transformers _tied_weights_keys list bug (Nemotron sets it as a list,
# transformers >=5.x calls .keys() on it)
import transformers.modeling_utils as _mu

def _patched_get_tied_weight_keys(module):
    for name, submodule in module.named_modules():
        tied = getattr(submodule, '_tied_weights_keys', {}) or {}
        if isinstance(tied, list):
            tied = {k: None for k in tied}
        yield from ([f'{name}.{k}' if name else k for k in tied.keys()])

_mu._get_tied_weight_keys = _patched_get_tied_weight_keys
print('Patched _get_tied_weight_keys.')

print(f'VRAM before training: {torch.cuda.memory_allocated() / 1e9:.1f} GB')
print('Starting SFT...')
trainer.train(resume_from_checkpoint=resume_from)
print('\nSFT complete!')

# Find best checkpoint by eval_loss and store in BEST_SFT_CKPT for Cell 8
BEST_SFT_CKPT = None
best_loss = float('inf')
for state_file in sorted(glob.glob(f'{CKPT_DIR}/checkpoint-*/trainer_state.json')):
    try:
        with open(state_file) as f:
            ts = json.load(f)
        for entry in ts.get('log_history', []):
            if 'eval_loss' in entry and entry['eval_loss'] < best_loss:
                best_loss = entry['eval_loss']
                BEST_SFT_CKPT = os.path.dirname(state_file)
    except Exception:
        pass

if BEST_SFT_CKPT is None:
    ckpts = sorted(glob.glob(f'{CKPT_DIR}/checkpoint-*'))
    BEST_SFT_CKPT = ckpts[-1] if ckpts else CKPT_DIR
    best_loss = float('nan')

print(f'Best checkpoint: {BEST_SFT_CKPT}  (eval_loss={best_loss})')


## Cell 7 — GRPO Reward Fine-tuning *(optional — run after SFT)*

Two reward signals: correctness (1.0) and format (0.2).

In [ ]:
import re
from trl import GRPOTrainer, GRPOConfig
from transformers import TrainerCallback
from unsloth import FastLanguageModel
import torch

FastLanguageModel.for_training(model)
for name, param in model.named_parameters():
    if param.requires_grad and param.dtype != torch.bfloat16:
        param.data = param.data.to(torch.bfloat16)

class BFloat16RecastCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, model=None, **kwargs):
        if model is None:
            return
        cast_count = 0
        for name, param in model.named_parameters():
            if param.requires_grad and param.dtype != torch.bfloat16:
                param.data = param.data.to(torch.bfloat16)
                cast_count += 1
        if cast_count:
            print(f'[BFloat16RecastCallback] Re-cast {cast_count} params -> bfloat16.')

def extract_boxed(text):
    m = re.search(r'\\boxed\{', text)
    if not m:
        return ''
    start, depth = m.end(), 1
    for i, ch in enumerate(text[start:]):
        depth += (ch == '{') - (ch == '}')
        if depth == 0:
            return text[start:start + i].strip()
    return ''

def num_eq(a, b, tol=1e-6):
    try:
        fa = float(a.replace(',', ''))
        fb = float(b.replace(',', ''))
        return abs(fa - fb) / max(abs(fb), 1e-9) < tol
    except ValueError:
        return False

def reward_correctness(completions, solution=None, **kwargs):
    if solution is None:
        return [0.0] * len(completions)
    gold = extract_boxed(solution[0] if isinstance(solution, list) else solution)
    rewards = []
    for c in completions:
        pred = extract_boxed(c)
        if not pred:
            rewards.append(0.0)
        elif pred == gold or num_eq(pred, gold):
            rewards.append(1.0)
        else:
            rewards.append(0.1)
    return rewards

def reward_format(completions, **kwargs):
    return [0.2 if re.search(r'\\boxed\{', c) else 0.0 for c in completions]

def fmt_grpo(x):
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': x['problem']},
    ]
    return {
        'prompt':   tokenizer.apply_chat_template(
                        msgs, tokenize=False, add_generation_prompt=True),
        'solution': x['solution'],
    }

grpo_ds = grpo_raw.map(fmt_grpo, remove_columns=grpo_raw.column_names, desc='GRPO format')
print(f'GRPO dataset: {len(grpo_ds):,} examples')

grpo_cfg = GRPOConfig(
    output_dir='/kaggle/working/grpo_checkpoints',
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    max_grad_norm=0.3,
    warmup_ratio=0.1,
    bf16=True,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    report_to='none',
    num_generations=4,
    max_completion_length=1024,
    generation_kwargs={'temperature': 0.7, 'do_sample': True},
    seed=42,
)

grpo_trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_correctness, reward_format],
    args=grpo_cfg,
    train_dataset=grpo_ds,
    callbacks=[BFloat16RecastCallback()],
)

print('Starting GRPO...')
grpo_trainer.train()
print('\nGRPO complete!')


## Cell 8 — Package adapter for submission

Copies adapter files directly from the best checkpoint — **no `model.save_pretrained()` call**. This avoids the OOM crash. Works correctly whether Cell 6 completed or was interrupted mid-run.

In [ ]:
import os, shutil, json, glob, zipfile

# ── Find best checkpoint ──────────────────────────────────────────────────────
# BEST_SFT_CKPT is set by Cell 6 when training completes.
# If Cell 6 was interrupted, we find the best saved checkpoint automatically.
# Either way, we copy files directly — NO model.save_pretrained() call.
# (model.save_pretrained() re-allocates model weights into VRAM and caused OOM.)

CKPT_DIR = '/kaggle/working/checkpoints'

if 'BEST_SFT_CKPT' not in dir() or not (BEST_SFT_CKPT and os.path.isdir(BEST_SFT_CKPT)):
    print('BEST_SFT_CKPT not set by Cell 6 — scanning checkpoints...')
    best_ckpt, best_loss = None, float('inf')
    for state_file in sorted(glob.glob(f'{CKPT_DIR}/checkpoint-*/trainer_state.json')):
        try:
            with open(state_file) as f:
                ts = json.load(f)
            for entry in ts.get('log_history', []):
                if 'eval_loss' in entry and entry['eval_loss'] < best_loss:
                    best_loss = entry['eval_loss']
                    best_ckpt = os.path.dirname(state_file)
        except Exception:
            pass
    if best_ckpt is None:
        ckpts = sorted(glob.glob(f'{CKPT_DIR}/checkpoint-*'))
        assert ckpts, (
            'No checkpoints found in /kaggle/working/checkpoints/\n'
            'Run Cell 6 (SFT training) first.'
        )
        best_ckpt = ckpts[-1]
        best_loss = float('nan')
    BEST_SFT_CKPT = best_ckpt
    print(f'Using checkpoint: {BEST_SFT_CKPT}  (eval_loss={best_loss})')
else:
    print(f'Using BEST_SFT_CKPT from Cell 6: {BEST_SFT_CKPT}')

# ── Copy adapter files from checkpoint — no model reload needed ───────────────
ADAPTER_OUT = '/kaggle/working/nemotron-adapter-ready-to-submit'
if os.path.exists(ADAPTER_OUT):
    shutil.rmtree(ADAPTER_OUT)
os.makedirs(ADAPTER_OUT)

ADAPTER_FILES = {
    'adapter_config.json',
    'adapter_model.safetensors',
    'tokenizer.json',
    'tokenizer_config.json',
    'special_tokens_map.json',
    'tokenizer.model',
    'added_tokens.json',
}

copied = []
for fn in os.listdir(BEST_SFT_CKPT):
    if fn in ADAPTER_FILES or (fn.startswith('adapter_model') and fn.endswith('.safetensors')):
        shutil.copy2(os.path.join(BEST_SFT_CKPT, fn), os.path.join(ADAPTER_OUT, fn))
        copied.append(fn)

print(f'Copied {len(copied)} files from {BEST_SFT_CKPT}:')
for fn in sorted(copied):
    print(f'  {fn}')

# ── Verify required files ─────────────────────────────────────────────────────
for req in ('adapter_config.json', 'adapter_model.safetensors'):
    assert os.path.exists(os.path.join(ADAPTER_OUT, req)), (
        f'MISSING: {req}\n'
        f'The checkpoint at {BEST_SFT_CKPT} did not contain this file.\n'
        f'Ensure unsloth is saving adapter-only checkpoints (not full model shards).'
    )

# ── Verify rank ───────────────────────────────────────────────────────────────
with open(os.path.join(ADAPTER_OUT, 'adapter_config.json')) as f:
    cfg = json.load(f)
print('\nadapter_config.json:')
print(json.dumps(cfg, indent=2))

rank = cfg.get('r', 999)
assert rank <= 32, f'Rank {rank} exceeds competition limit of 32!'
print(f'\nRank check passed: r={rank} <= 32')

# ── Size summary ──────────────────────────────────────────────────────────────
print('\nFiles:')
total_mb = 0
for fn in sorted(os.listdir(ADAPTER_OUT)):
    sz = os.path.getsize(os.path.join(ADAPTER_OUT, fn)) / 1e6
    total_mb += sz
    print(f'  {fn:45s} {sz:6.1f} MB')
print(f'  {"TOTAL":45s} {total_mb:6.1f} MB')

adapter_sz = os.path.getsize(os.path.join(ADAPTER_OUT, 'adapter_model.safetensors')) / 1e6
if adapter_sz > 2000:
    print(f'\nWARNING: adapter_model.safetensors is {adapter_sz:.0f} MB — looks like a merged model.')
    print('Expected a LoRA adapter (~100-600 MB). Check unsloth checkpoint save settings.')
else:
    print(f'\nAdapter size OK: {adapter_sz:.0f} MB')


## Cell 9 — Zip and submit

In [ ]:
import shutil, os, zipfile

for req in ('adapter_config.json', 'adapter_model.safetensors'):
    assert os.path.exists(os.path.join(ADAPTER_OUT, req)), \
        f'MISSING {req} in {ADAPTER_OUT} — run Cell 8 first.'

ZIP_PATH = '/kaggle/working/submission.zip'
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

# Write each file at the zip root (no subdirectory prefix)
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for fn in sorted(os.listdir(ADAPTER_OUT)):
        zf.write(os.path.join(ADAPTER_OUT, fn), arcname=fn)

size_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f'Created : {ZIP_PATH}')
print(f'Size    : {size_mb:.1f} MB')

print('\nZip contents:')
with zipfile.ZipFile(ZIP_PATH) as zf:
    for info in zf.infolist():
        print(f'  {info.filename:45s} {info.file_size / 1e6:6.1f} MB')

print('\nSubmit /kaggle/working/submission.zip to the competition.')
print('Done')
